[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/125/blob/main/07_dataframes/07_4_Groups_and_Crosstabs.ipynb)

# 07.4: Groups and Crosstabs

Most interesting questions compare groups. Do musicals earn more than plays? Which theatre brought in the most money? You could answer these with a mask for each group, but that means writing one mask per group, and this data has 56 theatres. This notebook introduces `.groupby()`, which does the work for every group at once, and `pd.crosstab()`, which counts combinations of two categories.

## The data

We use the Broadway data again, where each row is one show in one week.

In [ ]:
import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_corgis_broadway.csv")
df.head(2)

## The problem with masks

Here is one comparison done with masks.

In [ ]:
print(df[df["Type"] == "Musical"]["Gross"].mean())
print(df[df["Type"] == "Play"]["Gross"].mean())
print(df[df["Type"] == "Special"]["Gross"].mean())

This works, but it repeats the same code for each group.

## `.groupby()`

`.groupby()` splits the rows into groups based on the values in a column, calculates something for each group, and combines the answers into one result. You write three things: the column that defines the groups, the column to calculate on, and the calculation.

In [ ]:
df.groupby("Type")["Gross"].mean()

These are the same three numbers as above. A musical takes in about 700312 in a typical week, a special about 365322 and a play about 325718. Read the code from left to right: "group by Type, look at Gross, and take the mean." The result is a series, where the index holds the groups, so you can use everything from module 06 on it. Any of the statistics you know will work here, including `.median()`, `.sum()`, `.min()`, `.max()` and `.count()`.

In [ ]:
df.groupby("Type")["Gross"].count()

Always check how many rows are behind a group's average. Specials have a similar mean to plays, but that mean is based on only 339 rows compared with 8406 for plays.

## Sorting a grouped result

Since the result is a series, `.sort_values()` and `.head()` give you a top-N list. Which shows brought in the most money in total?

In [ ]:
df.groupby("Name")["Gross"].sum().sort_values(ascending=False).head(3)

The Lion King leads with about 1.29 billion dollars, followed by Wicked at about 1.04 billion and The Phantom Of The Opera at about 0.79 billion. Now compare that with a different column to group by. Which theatres brought in the most?

In [ ]:
df.groupby("Theatre")["Gross"].sum().sort_values(ascending=False).head(3)

The Gershwin, where Wicked plays, leads with about 1.22 billion. Each of these took one line, and they answer questions that would have needed a mask for each of 820 shows or 56 theatres.

## More than one column

To calculate on several columns, pass a list of column names. The result is a DataFrame with one row per group.

In [ ]:
df.groupby("Type")[["Gross", "Attendance"]].mean()

Musicals lead on both, with a mean attendance of about 9131 per week against about 5122 for plays. Notice that `Type` is in the index of this result. If you want it back as an ordinary column, which is what you will need for making charts in module 08, use `.reset_index()`.

In [ ]:
df.groupby("Type")["Gross"].mean().reset_index()

The result is now a small DataFrame with two ordinary columns, `Type` and `Gross`. In this course we always group by one column at a time. When a question involves two categories at once, we use a crosstab.

## Crosstabs: counting two categories at once

`pd.crosstab()` counts how many rows fall into each combination of two categories. To have a second category to work with, we can make one with a mask: was this a week with a gross of more than a million dollars?

In [ ]:
df["Million_Week"] = df["Gross"] > 1000000
pd.crosstab(df["Type"], df["Million_Week"])

Each number counts rows. The first argument gives the rows of the table and the second gives the columns, so musicals had 3906 million-dollar weeks and plays had 175. Raw counts are hard to compare here because there are so many more musical rows than play rows. `normalize="index"` converts each row into fractions that add up to 1.

In [ ]:
pd.crosstab(df["Type"], df["Million_Week"], normalize="index")

Now the rows can be compared directly. About 17 percent of musical weeks reached a million dollars, compared with about 9 percent for specials and about 2 percent for plays. Be careful about what each fraction is a share of: `normalize="index"` gives the share within each row, so 0.173 is the fraction of musical weeks, and not the fraction of million-dollar weeks that were musicals. `margins=True` adds a row and a column of totals to a table of counts.

In [ ]:
pd.crosstab(df["Type"], df["Million_Week"], margins=True)

The `All` row and column give the totals: 4111 of the 31296 rows are million-dollar weeks.

## Making categories from numbers: `pd.cut()`

A crosstab needs categories. `pd.cut()` makes categories from a numeric column. You give it the edges of the ranges in `bins` and a name for each range in `labels`. By default each range includes its upper edge and not its lower edge, so 1989 to 1999 covers the years 1990 through 1999.

In [ ]:
df["Period"] = pd.cut(df["Year"], bins=[1989, 1999, 2009, 2016], labels=["1990s", "2000s", "2010s"])
pd.crosstab(df["Period"], df["Type"], normalize="index")

Reading across each row gives the mix of show types in that period. Musicals grew from about 69 percent of show-weeks in the 1990s to about 74 percent in the 2010s, while plays fell from about 29 percent to about 26 percent. Note that the data stops in August of 2016, so the 2010s row covers fewer years than the others, but fractions make that less of a problem than counts would.

## What's next

That finishes module 07. You can now load a dataset, check it, select from it and compare groups. All of your results so far have been tables of numbers, though, and some patterns are much easier to see in a picture. Module 08 starts with notebook 08.1, which shows how to make charts.